# Categorical Encoding

Machine Learning models are ultimately just massive mathematical equations. If you try to multiply a weight by "New York" or "Premium Subscription", the math breaks. The model only understands numbers.

**Categorical Encoding** is the process of converting text categories into numeric values. However, you cannot just randomly assign numbers to words. You must first identify what *type* of category you are dealing with:

1. **Ordinal Data**: Categories that have a logical, built-in order (e.g., *Low, Medium, High* or *Bronze, Silver, Gold*).
2. **Nominal Data**: Categories that have NO logical order (e.g., *Red, Blue, Green* or *New York, London, Paris*).

Let's set up a Python sandbox to see why this distinction is so important!

In [1]:
import pandas as pd

# Create a dataset with both Ordinal and Nominal text data
data = {
    'customer_id': [1, 2, 3, 4, 5],
    'city': ['London', 'New York', 'Paris', 'London', 'Paris'], # Nominal (No order)
    'subscription': ['Basic', 'Premium', 'VIP', 'Basic', 'Premium'], # Ordinal (Ordered)
    'spend': [150, 400, 1200, 100, 500]
}

df = pd.DataFrame(data)

print("--- Original Text Data ---")
display(df)

--- Original Text Data ---


,customer_id,city,subscription,spend
0,1,London,Basic,150
1,2,New York,Premium,400
2,3,Paris,VIP,1200
3,4,London,Basic,100
4,5,Paris,Premium,500


# 1. Ordinal Encoding (Order Matters)
Because our `subscription` column has a natural hierarchy (`Basic < Premium < VIP`), we *want* the model to know that VIP is "greater" or "higher" than Basic. 

We can do this easily in Pandas using the `.map()` function to assign ascending numbers to the categories.

In [2]:
# Create a copy
df_ordinal = df.copy()

# 1. Define the mapping dictionary (lowest to highest)
subscription_map = {
    'Basic': 1,
    'Premium': 2,
    'VIP': 3
}

# 2. Apply the map to the column
df_ordinal['subscription_encoded'] = df_ordinal['subscription'].map(subscription_map)

print("--- Data after Ordinal Encoding ---")
display(df_ordinal[['customer_id', 'subscription', 'subscription_encoded']])

--- Data after Ordinal Encoding ---


,customer_id,subscription,subscription_encoded
0,1,Basic,1
1,2,Premium,2
2,3,VIP,3
3,4,Basic,1
4,5,Premium,2


*(Now the model sees exactly what we see: VIP (3) is mathematically larger than Basic (1)!)*

# 2. The Danger of Label Encoding Nominal Data
What if we do the exact same thing to our `city` column?
* London = 1
* New York = 2
* Paris = 3

**This is a critical mistake!** If we feed this to a model, the algorithm will mathematically assume that `Paris (3)` is three times larger, or three times more important, than `London (1)`. It will also assume that the "average" of London and Paris is New York (`(1 + 3) / 2 = 2`). This is complete nonsense!

To handle Nominal data, we must use a different technique.

# 3. One-Hot Encoding (Order Doesn't Matter)
To prevent the model from assuming an order, we use **One-Hot Encoding**. 

This technique creates a brand new column for *every single unique category* in the original column. It then places a `1` if the category is present, and a `0` if it is not. 

The easiest way to do this for quick analysis is using Pandas' `pd.get_dummies()`.

In [3]:
# Apply One-Hot Encoding to the 'city' column
df_onehot = pd.get_dummies(df, columns=['city'])

print("--- Data after One-Hot Encoding (Pandas) ---")
display(df_onehot)

--- Data after One-Hot Encoding (Pandas) ---


,customer_id,subscription,spend,city_London,city_New York,city_Paris
0,1,Basic,150,True,False,False
1,2,Premium,400,False,True,False
2,3,VIP,1200,False,False,True
3,4,Basic,100,True,False,False
4,5,Premium,500,False,False,True


*(Notice how the 'city' column vanished, and was replaced by 'city_London', 'city_New York', and 'city_Paris'? Now, Paris is just a `1` in the Paris column, and London is just a `1` in the London column. They are mathematically equal!)*

# 4. The Dummy Variable Trap (`drop_first`)
There is a slight mathematical issue with basic One-Hot Encoding called the **Dummy Variable Trap** (or Multicollinearity). 

If a customer is a `0` in London, and a `0` in New York, the model can logically deduce with 100% certainty that they *must* be a `1` in Paris (because there are only 3 cities). 

Having a 'city_Paris' column is perfectly redundant. In algorithms like Linear Regression, this redundancy can actually break the math. To fix this, we drop the first dummy column.

In [4]:
# Drop the first category to prevent the Dummy Variable Trap
df_trap_fixed = pd.get_dummies(df, columns=['city'], drop_first=True)

print("--- Data after dropping the first dummy ---")
display(df_trap_fixed)

--- Data after dropping the first dummy ---


,customer_id,subscription,spend,city_New York,city_Paris
0,1,Basic,150,False,False
1,2,Premium,400,True,False
2,3,VIP,1200,False,True
3,4,Basic,100,False,False
4,5,Premium,500,False,True


*(London is gone! But the data is perfectly intact. If a row has a `0` for New York and a `0` for Paris, the model knows it represents London. We just saved memory and prevented mathematical errors!)*

# 5. One-Hot Encoding for Production (Scikit-Learn)
Just like with missing values and scaling, if you are building a real Machine Learning pipeline, you shouldn't use Pandas for encoding. You should use **Scikit-Learn's `OneHotEncoder`**. 

Why? If your testing dataset doesn't happen to have anyone from Paris in it, Pandas `get_dummies` won't create a Paris column, and your columns won't match your training data! Scikit-Learn *remembers* the categories so your columns always match.

In [5]:
from sklearn.preprocessing import OneHotEncoder

# 1. Initialize the Encoder
# sparse_output=False ensures it returns a standard array instead of a compressed matrix
# drop='first' prevents the Dummy Variable Trap
encoder = OneHotEncoder(sparse_output=False, drop='first')

# 2. Fit and Transform the 'city' column
# Note: Scikit-Learn expects a 2D array, so we pass df[['city']]
encoded_array = encoder.fit_transform(df[['city']])

# 3. (Optional) Turn the output array back into a pretty Pandas DataFrame
# encoder.get_feature_names_out() gets the correct column names!
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(['city']))

print("--- Scikit-Learn Output ---")
display(encoded_df)

--- Scikit-Learn Output ---


,city_New York,city_Paris
0,0.0,0.0
1,1.0,0.0
2,0.0,1.0
3,0.0,0.0
4,0.0,1.0


## Real-World Use Case or Analogy:
Think of Categorical Encoding like ordering at a **Fast Food Drive-Thru**:

* **Ordinal Encoding (Drink Size)**: You order a soda. The sizes are Small, Medium, and Large. These have a clear order. It makes perfect sense for the cash register to ring these up as `Size 1`, `Size 2`, and `Size 3`. Size 3 has mathematically more ounces of liquid than Size 1.
* **One-Hot Encoding (Burger Condiments)**: You are asked what you want on your burger: Ketchup, Mustard, or Mayo. 
    * If the cashier typed Ketchup as 1, Mustard as 2, and Mayo as 3, the kitchen might look at a ticket for "2" and wonder if you want Mustard, or if you want *two portions of Ketchup*. 
    * Instead, the kitchen ticket has three separate checkboxes: `[ ] Ketchup`, `[X] Mustard`, `[ ] Mayo`. This is exactly what One-Hot Encoding does. It separates unordered choices into independent Yes/No (1/0) checkboxes so there is zero confusion!

---